# 11 – Retry & Resilience

Two retry mechanisms:
- **`@with_retry`** — decorator with exponential backoff for any function
- **`retry_agent_call`** — wraps agent `.execute()`, returns `AgentResult(success=False)` on final failure instead of raising

Both use configurable `max_retries` and `backoff_factor`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from core.retry import with_retry, retry_agent_call
from core.base_agent import AgentRequest, AgentResult
from unittest.mock import patch

## 1. @with_retry — succeeds on first try

In [ ]:
calls = []

@with_retry(max_retries=3, backoff_factor=0.001)
def reliable_fn():
    calls.append('called')
    return 'success'

result = reliable_fn()
print(f'Result: {result}')
print(f'Total calls: {len(calls)}')

## 2. @with_retry — fails twice, succeeds on 3rd attempt

In [ ]:
attempt = [0]

@with_retry(max_retries=3, backoff_factor=0.001)
def flaky_fn():
    attempt[0] += 1
    if attempt[0] < 3:
        raise ConnectionError(f'attempt {attempt[0]} failed')
    return f'success on attempt {attempt[0]}'

with patch('time.sleep'):  # skip actual sleep
    result = flaky_fn()

print(f'Result: {result}')
print(f'Total attempts: {attempt[0]}')

## 3. @with_retry — exhausts all retries, raises exception

In [ ]:
calls2 = []

@with_retry(max_retries=3, backoff_factor=0.001)
def always_fails():
    calls2.append(1)
    raise RuntimeError('always broken')

try:
    with patch('time.sleep'):
        always_fails()
except RuntimeError as e:
    print(f'Exception raised: {e}')
    print(f'Total attempts: {len(calls2)}')

## 4. retry_agent_call — safe wrapper (returns AgentResult on failure)

In [ ]:
from agents.information_agent import InformationAgent

agent = InformationAgent()
req = AgentRequest(query='retention metrics', data_products=['retention'])

# Successful call
result = retry_agent_call(agent.execute, req, max_retries=3)
print('Success:', result.success)
print('Message:', result.message)

## 5. retry_agent_call — agent that always raises

In [ ]:
from core.base_agent import BaseAgent

class BrokenAgent(BaseAgent):
    @property
    def name(self): return 'broken_agent'
    def execute(self, request):
        raise ConnectionError('Databricks unreachable')

broken = BrokenAgent()
req = AgentRequest(query='test')

with patch('time.sleep'):
    result = retry_agent_call(broken.execute, req, max_retries=3)

print('Success  :', result.success)
print('Message  :', result.message)
print('Errors   :', result.errors)
# No exception was raised — safe degradation

## 6. Selective exception types

In [ ]:
calls3 = []

# Only retry on ConnectionError, not ValueError
@with_retry(max_retries=3, backoff_factor=0.001, exceptions=(ConnectionError,))
def selective_retry():
    calls3.append(1)
    raise ValueError('not a connection error')  # should NOT be retried

try:
    with patch('time.sleep'):
        selective_retry()
except ValueError as e:
    print(f'ValueError raised immediately: {e}')
    print(f'Attempts (should be 1): {len(calls3)}')